# SwissCrawl GAHD stratified sampling

Run this notebook in Google Colab with a GPU runtime. It scores the complete SwissCrawl candidate pool with the official GAHD classifier, saves resumable checkpoints in Google Drive, and produces the fixed 700-row AustroTox-adapted sample. Haiku is not called here.

Before running, select **Runtime > Change runtime type > GPU**. The notebook will ask you to upload `swisscrawl_filtered.csv` and mount Google Drive.

In [ ]:
%pip install -q pandas==2.3.3 numpy==2.2.6 transformers==5.18.0 huggingface_hub==1.33.0 safetensors

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib
import json
import sqlite3
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = "jagoldz/gahd"
REVISION = "90c6e4f50d109c42bebdeb0f43fb5c3e619612fc"
BATCH_SIZE = 32
RANDOM_SEED = 42
STRATA = ("0.0-0.3", "0.3-0.5", "0.5-0.7", "0.7-0.9", "0.9-1.0")
TARGETS = {"0.0-0.3": 134, "0.3-0.5": 134, "0.5-0.7": 134, "0.7-0.9": 164, "0.9-1.0": 134}

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/swisscrawl_gahd")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if len(csv_names) != 1:
    raise ValueError("Upload exactly one CSV file: swisscrawl_filtered.csv")
candidate_path = Path("/content") / csv_names[0]
candidate = pd.read_csv(candidate_path, dtype={"sentence_id": "string"})
required = {"sentence_id", "text"}
if not required.issubset(candidate.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(candidate.columns))}")
if candidate["sentence_id"].isna().any() or candidate["sentence_id"].duplicated().any():
    raise ValueError("sentence_id values must be unique and non-null")
if not candidate["text"].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
    raise ValueError("Every text must be a non-empty string")
print(f"Loaded {len(candidate):,} candidates")

In [ ]:
def corpus_fingerprint(frame):
    digest = hashlib.sha256()
    schema = [(str(column), str(dtype)) for column, dtype in frame.dtypes.items()]
    digest.update(json.dumps(schema, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))
    digest.update(pd.util.hash_pandas_object(frame, index=True).values.tobytes())
    return digest.hexdigest()

checkpoint_path = OUTPUT_DIR / "swisscrawl_gahd_scores.sqlite3"
connection = sqlite3.connect(checkpoint_path)
connection.executescript(
    """
    CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS scores (
        sentence_id TEXT PRIMARY KEY,
        hate_score REAL NOT NULL CHECK (hate_score >= 0 AND hate_score <= 1)
    );
    """
)
expected_metadata = {
    "corpus_hash": corpus_fingerprint(candidate),
    "model": MODEL,
    "revision": REVISION,
}
existing_metadata = dict(connection.execute("SELECT key, value FROM metadata"))
if existing_metadata and existing_metadata != expected_metadata:
    connection.close()
    raise ValueError("The Drive checkpoint belongs to another corpus or model revision.")
if not existing_metadata:
    with connection:
        connection.executemany("INSERT INTO metadata(key, value) VALUES (?, ?)", expected_metadata.items())

completed = {row[0] for row in connection.execute("SELECT sentence_id FROM scores")}
pending = candidate.loc[~candidate["sentence_id"].isin(completed), ["sentence_id", "text"]]
print(f"Checkpoint: {len(completed):,} complete, {len(pending):,} pending")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, revision=REVISION, use_safetensors=True
).to("cuda")
model.eval()

started = time.perf_counter()
processed = 0
for start in range(0, len(pending), BATCH_SIZE):
    batch = pending.iloc[start : start + BATCH_SIZE]
    encoded = tokenizer(
        batch["text"].tolist(),
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    rows = [(str(sentence_id), float(score)) for sentence_id, score in zip(batch["sentence_id"], probabilities)]
    with connection:
        connection.executemany(
            "INSERT OR REPLACE INTO scores(sentence_id, hate_score) VALUES (?, ?)", rows
        )
    processed += len(batch)
    if processed % (BATCH_SIZE * 50) == 0 or processed == len(pending):
        elapsed = time.perf_counter() - started
        total_complete = len(completed) + processed
        print(f"{total_complete:,}/{len(candidate):,} ({total_complete / len(candidate):.1%}), {processed / elapsed:.1f} rows/s")

score_by_id = dict(connection.execute("SELECT sentence_id, hate_score FROM scores"))
connection.close()
if len(score_by_id) != len(candidate):
    raise RuntimeError(f"Checkpoint has {len(score_by_id):,} scores, expected {len(candidate):,}")
scored = candidate.copy()
scored["hate_score"] = scored["sentence_id"].map(score_by_id)
scored_path = OUTPUT_DIR / "swisscrawl_scored_gahd.csv"
scored.to_csv(scored_path, index=False)
print("Saved", scored_path)

In [ ]:
def assign_strata(scores):
    values = pd.Series(scores, dtype=float)
    if values.isna().any() or not values.between(0, 1).all():
        raise ValueError("Scores must all be between 0 and 1")
    labels = np.select(
        [values < 0.3, values < 0.5, values < 0.7, values < 0.9],
        STRATA[:-1],
        default=STRATA[-1],
    )
    return pd.Series(pd.Categorical(labels, categories=STRATA, ordered=True), index=values.index)

scored["austrotox_stratum"] = assign_strata(scored["hate_score"]).to_numpy()
source_counts = scored["austrotox_stratum"].value_counts(sort=False)
shortages = [
    f"{stratum}: available={int(source_counts.get(stratum, 0))}, required={target}"
    for stratum, target in TARGETS.items()
    if int(source_counts.get(stratum, 0)) < target
]
if shortages:
    raise ValueError("Undersized strata: " + "; ".join(shortages))

parts = [
    scored[scored["austrotox_stratum"] == stratum].sample(n=target, random_state=RANDOM_SEED)
    for stratum, target in TARGETS.items()
]
master = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
selected_counts = master["austrotox_stratum"].value_counts(sort=False)
report = {
    "method": "AustroTox-inspired project-specific fixed allocation",
    "sample_size": int(len(master)),
    "random_seed": RANDOM_SEED,
    "score_model": MODEL,
    "score_revision": REVISION,
    "source_counts": {key: int(source_counts.get(key, 0)) for key in TARGETS},
    "target_counts": TARGETS,
    "selected_counts": {key: int(selected_counts.get(key, 0)) for key in TARGETS},
    "score_summaries": {
        key: {
            "min": float(scored.loc[scored["austrotox_stratum"] == key, "hate_score"].min()),
            "median": float(scored.loc[scored["austrotox_stratum"] == key, "hate_score"].median()),
            "max": float(scored.loc[scored["austrotox_stratum"] == key, "hate_score"].max()),
        }
        for key in TARGETS
    },
}
master_path = OUTPUT_DIR / "master_sample_stratified.csv"
report_path = OUTPUT_DIR / "stratified_sampling_report.json"
master.to_csv(master_path, index=False)
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
assert len(master) == 700 and master["sentence_id"].nunique() == 700
assert report["selected_counts"] == TARGETS
print(json.dumps(report, indent=2))
print("Saved", master_path)
print("Saved", report_path)

After completion, download `master_sample_stratified.csv`, `stratified_sampling_report.json`, and optionally `swisscrawl_scored_gahd.csv` from `MyDrive/swisscrawl_gahd/`. Copy them into the local repository's `data/` directory. The local pipeline can then continue with Haiku translation of only the selected 700 rows.